In [9]:
"""
SEC EDGAR 10-Q Downloader + PDF Converter
==========================================
1. Fetches all 10-Q filings (2021-03-27 → 2026-03-27) from SEC EDGAR
2. Downloads each filing as HTML
3. Converts each HTML → PDF using xhtml2pdf (pure Python, no external dependencies)
4. Optionally deletes HTML after conversion
5. Saves a full manifest CSV

Output structure:
    ./10Q_filings/
        MSFT/
            MSFT_10Q_2024-07-25_0000789019-24-000083.pdf
            ...
        AAPL/
            ...
    10Q_filings_manifest.csv

Requirements:
    pip install requests pandas xhtml2pdf
"""

import os
import time
import requests
import pandas as pd
from xhtml2pdf import pisa
from datetime import datetime
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────

CUTOFF_START       = datetime(2021, 3, 27)
CUTOFF_END         = datetime(2026, 3, 27)

OUTPUT_DIR         = Path("10Q_filings")
MANIFEST_CSV       = "10Q_filings_manifest.csv"

DELETE_HTML        = True    # Remove .htm after successful PDF conversion
SKIP_IF_PDF_EXISTS = True    # Skip re-download + re-convert if PDF already exists

SLEEP_EDGAR        = 0.15    # Pause between EDGAR API calls (SEC max: 10 req/s)
SLEEP_DOWNLOAD     = 0.20    # Pause between file downloads
SLEEP_TICKER       = 0.50    # Pause between tickers

HEADERS = {
    "User-Agent":      "Ana Del Campo ana@microsoft.com",  # ← your name + email
    "Accept-Encoding": "gzip, deflate",
    "Accept":          "application/json",
}

# ── Ticker → CIK Map ──────────────────────────────────────────────────────────

TICKER_CIK = {
    "MSFT": (789019,   "Microsoft Corp"),
    "AAPL": (320193,   "Apple Inc"),
    "VZ":   (732712,   "Verizon Communications Inc"),
    "SOFI": (1818201,  "SoFi Technologies Inc"),
    "AGNC": (1412093,  "AGNC Investment Corp"),
    "ELPW": (1838987,  "Electrip Global"),            # ⚠ CIK unverified
    "U":    (1579428,  "Unity Software Inc"),
    "GME":  (1326380,  "GameStop Corp"),
    "NBIS": (1580144,  "Nebius Group NV"),             # ⚠ Formerly YNDX
    "AMAT": (796343,   "Applied Materials Inc"),
    "APP":  (1440681,  "Applovin Corp"),
    "RBLX": (1315098,  "Roblox Corp"),
    "SHOP": (1594805,  "Shopify Inc"),                 # ⚠ Some filings may be 6-K
    "LRCX": (707549,   "Lam Research Corp"),
    "ROKU": (1428439,  "Roku Inc"),
    "DKNG": (1801144,  "DraftKings Inc"),
    "CL":   (21665,    "Colgate-Palmolive Co"),
    "CHWY": (1766502,  "Chewy Inc"),
}

In [10]:
# Verify xhtml2pdf is available
pdf_ok = True
try:
    from xhtml2pdf import pisa
    import xhtml2pdf
    pdf_version = f"v{xhtml2pdf.__version__}"
except ImportError:
    pdf_ok = False
    pdf_version = "NOT FOUND"

print(f"\n{'='*65}")
print(f"  SEC EDGAR 10-Q Downloader + PDF Converter")
print(f"  Range      : {CUTOFF_START.date()} → {CUTOFF_END.date()}")
print(f"  Tickers    : {len(TICKER_CIK)}")
print(f"  Output     : {OUTPUT_DIR}/")
print(f"  xhtml2pdf  : {'✅ ' + pdf_version if pdf_ok else '❌ ' + pdf_version}")
print(f"  Delete HTML: {DELETE_HTML}")
print(f"{'='*65}\n")

if not pdf_ok:
    print("  ⚠  xhtml2pdf not found! Install it before running:")
    print("     pip install xhtml2pdf\n")
    print("  HTML files will still be downloaded.\n")


  SEC EDGAR 10-Q Downloader + PDF Converter
  Range      : 2021-03-27 → 2026-03-27
  Tickers    : 18
  Output     : 10Q_filings/
  xhtml2pdf  : ✅ v0.2.17
  Delete HTML: True



In [ ]:

# ── Step 1: EDGAR API — Fetch Submissions ─────────────────────────────────────

def get_submissions(cik: int) -> dict:
    """Fetch full submissions JSON for a CIK, merging paginated older filings."""
    cik_padded = str(cik).zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    data = r.json()
    time.sleep(SLEEP_EDGAR)

    # Merge any paginated older filing pages
    for extra_file in data["filings"].get("files", []):
        extra_url = f"https://data.sec.gov/submissions/{extra_file['name']}"
        re = requests.get(extra_url, headers=HEADERS, timeout=20)
        re.raise_for_status()
        extra = re.json()
        time.sleep(SLEEP_EDGAR)
        for key in ["form", "filingDate", "reportDate",
                    "accessionNumber", "primaryDocument"]:
            data["filings"]["recent"][key].extend(extra.get(key, []))

    return data


# ── Step 2: Parse 10-Q Filings Within Date Range ──────────────────────────────

def parse_10q_filings(data: dict, ticker: str, cik: int, company: str) -> list:
    """Extract 10-Q filings within the cutoff window from submissions data."""
    filings  = data["filings"]["recent"]
    forms    = filings.get("form", [])
    dates    = filings.get("filingDate", [])
    periods  = filings.get("reportDate", [])
    accnos   = filings.get("accessionNumber", [])
    primdocs = filings.get("primaryDocument", [])

    results = []
    for form, date_str, period, accno, primdoc in zip(
            forms, dates, periods, accnos, primdocs):

        if form != "10-Q":
            continue
        filing_date = datetime.strptime(date_str, "%Y-%m-%d")
        if not (CUTOFF_START <= filing_date <= CUTOFF_END):
            continue

        accno_nodash = accno.replace("-", "")
        doc_url = (
            f"https://www.sec.gov/Archives/edgar/data/{cik}/"
            f"{accno_nodash}/{primdoc}"
        )

        safe_accno = accno.replace("/", "_")
        ext        = Path(primdoc).suffix or ".htm"
        filename   = f"{ticker}_10Q_{date_str}_{safe_accno}{ext}"

        results.append({
            "Ticker":           ticker,
            "Company Name":     company,
            "CIK":              cik,
            "Filing Date":      date_str,
            "Period of Report": period,
            "Accession Number": accno,
            "Primary Document": primdoc,
            "Document URL":     doc_url,
            "HTML Filename":    filename,
            "HTML Path":        "",
            "PDF Path":         "",
            "Download Status":  "pending",
            "PDF Status":       "pending",
        })

    return results


# ── Step 3: Download HTML Filing ──────────────────────────────────────────────

def download_html(row: dict, output_dir: Path) -> dict:
    """Download a single 10-Q HTML filing to disk."""
    ticker    = row["Ticker"]
    filename  = row["HTML Filename"]
    url       = row["Document URL"]

    ticker_dir = output_dir / ticker
    ticker_dir.mkdir(parents=True, exist_ok=True)
    html_path  = ticker_dir / filename
    pdf_path   = html_path.with_suffix(".pdf")

    row["HTML Path"] = str(html_path)
    row["PDF Path"]  = str(pdf_path)

    # Skip entirely if PDF already exists
    if SKIP_IF_PDF_EXISTS and pdf_path.exists():
        row["Download Status"] = "skipped (PDF exists)"
        row["PDF Status"]      = "skipped (PDF exists)"
        return row

    # Skip HTML download if already on disk
    if html_path.exists():
        row["Download Status"] = "skipped (HTML exists)"
        return row

    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        html_path.write_bytes(r.content)
        row["Download Status"] = "downloaded"
        time.sleep(SLEEP_DOWNLOAD)
    except requests.HTTPError as e:
        row["Download Status"] = f"HTTP {e.response.status_code}"
    except Exception as e:
        row["Download Status"] = f"error: {e}"

    return row


# ── Step 4: Convert HTML → PDF via xhtml2pdf ──────────────────────────────────

def convert_to_pdf(row: dict) -> dict:
    """Convert downloaded HTML to PDF using xhtml2pdf."""

    # Skip if download failed
    if row["Download Status"] not in ("downloaded", "skipped (HTML exists)"):
        row["PDF Status"] = "skipped (no HTML)"
        return row

    # Skip if already converted
    if "skipped (PDF exists)" in row["PDF Status"]:
        return row

    html_path = Path(row["HTML Path"])
    pdf_path  = Path(row["PDF Path"])

    # Skip if PDF already on disk
    if SKIP_IF_PDF_EXISTS and pdf_path.exists():
        row["PDF Status"] = "skipped (PDF exists)"
        if DELETE_HTML and html_path.exists():
            html_path.unlink()
        return row

    try:
        # Read HTML content
        with open(html_path, 'rb') as html_file:
            html_content = html_file.read()
        
        # Convert HTML to PDF using xhtml2pdf with better error handling
        with open(pdf_path, 'wb') as pdf_file:
            pisa_status = pisa.CreatePDF(
                html_content,
                dest=pdf_file,
                encoding='utf-8',
                link_callback=None,
                default_css=None,
                xhtml=False,  # Don't require strict XHTML
                raise_exception=False  # Don't raise, just report errors
            )
        
        if pisa_status.err:
            row["PDF Status"] = f"partial conversion (errors={pisa_status.err})"
        else:
            row["PDF Status"] = "converted"
            
        # Delete HTML after conversion if flag is set (even with errors, PDF was created)
        if DELETE_HTML and html_path.exists():
            html_path.unlink()
            row["HTML Path"] = "deleted"

    except ImportError as e:
        # xhtml2pdf not installed
        row["PDF Status"] = f"xhtml2pdf not found: {e}"
    except Exception as e:
        row["PDF Status"] = f"conversion error: {type(e).__name__}: {str(e)[:50]}"

    return row


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    # Verify xhtml2pdf is available
    pdf_ok = True
    try:
        from xhtml2pdf import pisa
        import xhtml2pdf
        pdf_version = f"v{xhtml2pdf.__version__}"
    except ImportError:
        pdf_ok = False
        pdf_version = "NOT FOUND"

    print(f"\n{'='*65}")
    print(f"  SEC EDGAR 10-Q Downloader + PDF Converter")
    print(f"  Range      : {CUTOFF_START.date()} → {CUTOFF_END.date()}")
    print(f"  Tickers    : {len(TICKER_CIK)}")
    print(f"  Output     : {OUTPUT_DIR}/")
    print(f"  xhtml2pdf  : {'✅ ' + pdf_version if pdf_ok else '❌ ' + pdf_version}")
    print(f"  Delete HTML: {DELETE_HTML}")
    print(f"{'='*65}\n")

    if not pdf_ok:
        print("  ⚠  xhtml2pdf not found! Install it before running:")
        print("     pip install xhtml2pdf\n")
        print("  HTML files will still be downloaded.\n")

    all_rows = []
    errors   = []

    # ── STEP 1 & 2: Discover all 10-Q filings ────────────────────────────────
    print("[ STEP 1 ] Fetching filing metadata from EDGAR...\n")
    for ticker, (cik, company) in TICKER_CIK.items():
        try:
            data = get_submissions(cik)
            rows = parse_10q_filings(data, ticker, cik, company)
            all_rows.extend(rows)
            print(f"  ✓  {ticker:<6} {company:<35}  → {len(rows):>2} 10-Q filings found")
        except Exception as e:
            print(f"  ✗  {ticker:<6} {company:<35}  → ERROR: {e}")
            errors.append((ticker, str(e)))
        time.sleep(SLEEP_TICKER)

    total = len(all_rows)
    print(f"\n  Total filings to process : {total}")
    if errors:
        print(f"  Metadata errors          : {[t for t, _ in errors]}\n")

    if total == 0:
        print("\n  No filings found. Check CIKs or date range. Exiting.")
        return

    # ── STEP 3 & 4: Download + Convert ───────────────────────────────────────
    print(f"\n[ STEP 2 ] Downloading & converting {total} filings...\n")
    for i, row in enumerate(all_rows, 1):

        row = download_html(row, OUTPUT_DIR)
        row = convert_to_pdf(row)
        all_rows[i - 1] = row

        dl  = row["Download Status"]
        pdf = row["PDF Status"]

        if "error" in dl.lower() or "conversion error" in pdf.lower():
            icon = "✗"
        elif "converted" in pdf or "exists" in pdf or "partial" in pdf:
            icon = "✓"
        else:
            icon = "⏭"

        print(f"  {icon} [{i:>3}/{total}]  {row['Ticker']:<6}  "
              f"{row['Filing Date']}  "
              f"DL={dl:<25}  PDF={pdf}")

    # ── STEP 5: Save Manifest CSV ─────────────────────────────────────────────
    df = pd.DataFrame(all_rows)
    df.sort_values(["Ticker", "Filing Date"], ascending=[True, False], inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_csv(MANIFEST_CSV, index=False)

    # ── Final Summary ─────────────────────────────────────────────────────────
    downloaded  = (df["Download Status"] == "downloaded").sum()
    dl_skipped  = df["Download Status"].str.contains("skipped", na=False).sum()
    dl_errors   = df["Download Status"].str.contains("error|HTTP", na=False).sum()
    converted   = (df["PDF Status"] == "converted").sum()
    partial     = df["PDF Status"].str.contains("partial", na=False).sum()
    pdf_skipped = df["PDF Status"].str.contains("skipped", na=False).sum()
    pdf_errors  = df["PDF Status"].str.contains("conversion error", na=False).sum()

    print(f"\n{'='*65}")
    print(f"  DOWNLOADS")
    print(f"    ✅  Downloaded  : {downloaded}")
    print(f"    ⏭   Skipped    : {dl_skipped}  (already on disk)")
    print(f"    ❌  Failed      : {dl_errors}")
    print(f"  PDF CONVERSION")
    print(f"    ✅  Converted   : {converted}")
    print(f"    ⚠️   Partial    : {partial}  (converted with errors)")
    print(f"    ⏭   Skipped    : {pdf_skipped}  (already existed)")
    print(f"    ❌  Failed      : {pdf_errors}")
    print(f"  📄  Manifest     : {MANIFEST_CSV}")
    print(f"  📁  Output dir   : {OUTPUT_DIR}/")
    print(f"{'='*65}\n")

    if pdf_errors > 0:
        failed_pdf = df[df["PDF Status"].str.contains("conversion error", na=False)]
        print("  PDF conversion failures:")
        print(failed_pdf[["Ticker", "Filing Date", "PDF Status"]].to_string(index=False))
        print()
    
    if partial > 0:
        print(f"  ℹ️  {partial} filings converted with errors but PDFs were created")

    print("\nDone. ✅")


if __name__ == "__main__":
    main()